# AtmosIQ — LoRA V2 Adapter Inference on 25 Challenge Prompts

**Objective**: Run actual V2 adapter inference (`Qwen/Qwen2.5-0.5B-Instruct` + LoRA V2) on Google Colab's free T4 GPU across 25 realistic, ambiguous weather prompts.
- **Base Model**: `Qwen/Qwen2.5-0.5B-Instruct`
- **LoRA Adapter**: AtmosIQ Supervisor LoRA V2
- **Runtime**: Free Google Colab T4 GPU (~30 seconds execution)
- **Output**: Generates `atmosiq_lora_v2_challenge_report.json` and `atmosiq_lora_v2_challenge_predictions.json`


In [ ]:
# Step 1: Install compatible ML dependencies
!pip install -q torch transformers peft accelerate datasets


In [ ]:
# Step 2: Upload or extract LoRA V2 adapter
import os, zipfile, json
from pathlib import Path

# If uploading zip into Colab:
if os.path.exists('atmosiq_lora_v2_artifacts.zip'):
    with zipfile.ZipFile('atmosiq_lora_v2_artifacts.zip', 'r') as z:
        z.extractall('.')
    ADAPTER_DIR = 'atmosiq-supervisor-lora-v2'
elif os.path.exists('backend/models/atmosiq-supervisor-lora-v2'):
    ADAPTER_DIR = 'backend/models/atmosiq-supervisor-lora-v2'
elif os.path.exists('atmosiq-supervisor-lora-v2'):
    ADAPTER_DIR = 'atmosiq-supervisor-lora-v2'
else:
    ADAPTER_DIR = './atmosiq-supervisor-lora-v2'

print(f'Using adapter directory: {ADAPTER_DIR}')
assert os.path.exists(ADAPTER_DIR), f'Adapter directory not found at {ADAPTER_DIR}! Please upload atmosiq_lora_v2_artifacts.zip or the adapter directory.'


In [ ]:
# Step 3: Define 25 Challenge Prompts with Ground Truth
CHALLENGE_DATA = [
  {"id": 1, "text": "Is the heat unbearable outside in Ahmedabad right now?", "expected_route": "current_weather", "category": "current_weather", "notes": "Subjective sensation about current temperature"},
  {"id": 2, "text": "Current weather conditions in Chennai — is it raining at this moment?", "expected_route": "current_weather", "category": "current_weather", "notes": "Current rain status with punctuation and phrasing variation"},
  {"id": 3, "text": "What is the humidity level in Kolkata this afternoon?", "expected_route": "current_weather", "category": "current_weather", "notes": "Current afternoon humidity reading"},
  {"id": 4, "text": "Is there heavy wind blowing in Vizag right now?", "expected_route": "current_weather", "category": "current_weather", "notes": "Real-time wind observation"},
  {"id": 5, "text": "Current temperature and cloud cover in Srinagar at this hour", "expected_route": "current_weather", "category": "current_weather", "notes": "Multi-parameter real-time weather query"},
  {"id": 6, "text": "Will it rain this Sunday in Shimla?", "expected_route": "forecast", "category": "forecast", "notes": "Specific upcoming day precipitation forecast"},
  {"id": 7, "text": "Can I expect strong thunderstorms across Delhi over the next 48 hours?", "expected_route": "forecast", "category": "forecast", "notes": "48-hour severe weather forecast"},
  {"id": 8, "text": "What does the 7-day temperature trend look like for Jaipur?", "expected_route": "forecast", "category": "forecast", "notes": "Extended 7-day multi-day trend"},
  {"id": 9, "text": "Is precipitation expected tomorrow morning in Panvel?", "expected_route": "forecast", "category": "forecast", "notes": "Tomorrow morning specific window"},
  {"id": 10, "text": "Give me the hourly rain probability for Mumbai starting tonight", "expected_route": "forecast", "category": "forecast", "notes": "Hourly progression forecast"},
  {"id": 11, "text": "What precautions should elderly citizens take during this winter wave in Chandigarh?", "expected_route": "weather_advice", "category": "weather_advice", "notes": "Health and demographic-specific weather precautions"},
  {"id": 12, "text": "Is it safe to go cycling outdoors in heavy afternoon humidity in Nagpur?", "expected_route": "weather_advice", "category": "weather_advice", "notes": "Outdoor activity safety advice given weather condition"},
  {"id": 13, "text": "Should I delay hanging my laundry outside given today's sky in Bengaluru?", "expected_route": "weather_advice", "category": "weather_advice", "notes": "Domestic activity decision advice based on weather"},
  {"id": 14, "text": "What should a tourist pack for a high-altitude trip to Ladakh this week?", "expected_route": "weather_advice", "category": "weather_advice", "notes": "Packing and clothing advice for weather"},
  {"id": 15, "text": "Do I need sunscreen and UV protection for a 2-hour outdoor run today in Goa?", "expected_route": "weather_advice", "category": "weather_advice", "notes": "Sun protection and exercise advice"},
  {"id": 16, "text": "What is the real-time PM2.5 and PM10 reading around Anand Vihar station in Delhi?", "expected_route": "air_quality", "category": "air_quality", "notes": "Specific monitoring station particulate matter reading"},
  {"id": 17, "text": "Is the air in Kanpur hazardous to breathe today according to CPCB standards?", "expected_route": "air_quality", "category": "air_quality", "notes": "Health hazard query referring to CPCB scale"},
  {"id": 18, "text": "Check the National Air Quality Index category for Patna right now", "expected_route": "air_quality", "category": "air_quality", "notes": "National AQI category lookup"},
  {"id": 19, "text": "How bad is the particulate matter smog in Lucknow this morning?", "expected_route": "air_quality", "category": "air_quality", "notes": "Morning smog and particulate severity"},
  {"id": 20, "text": "Are sulfur dioxide and ozone levels elevated in Hyderabad today?", "expected_route": "air_quality", "category": "air_quality", "notes": "Gaseous air pollutant concentration query"},
  {"id": 21, "text": "What physical process causes radiation fog to form in valleys during winter?", "expected_route": "knowledge", "category": "knowledge", "notes": "Atmospheric physics explanation"},
  {"id": 22, "text": "Why does high relative humidity impair the human body's evaporative cooling?", "expected_route": "knowledge", "category": "knowledge", "notes": "Biometeorology scientific concept"},
  {"id": 23, "text": "How do Western Disturbances bring winter rain to northern India?", "expected_route": "knowledge", "category": "knowledge", "notes": "Indian climatology meteorological phenomenon"},
  {"id": 24, "text": "What is the difference between air quality index (AQI) and pollutant concentration (ug/m3)?", "expected_route": "knowledge", "category": "knowledge", "notes": "Technical index vs mass concentration concept"},
  {"id": 25, "text": "Explain how the urban heat island effect amplifies nighttime temperatures in cities", "expected_route": "knowledge", "category": "knowledge", "notes": "Urban climatology scientific mechanism"}
]
print(f'Loaded {len(CHALLENGE_DATA)} challenge prompts across 5 routes.')


In [ ]:
# Step 4: Load Base Model and LoRA V2 Adapter on GPU
import re, torch
from collections import Counter, defaultdict
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
VALID_ROUTES = {'current_weather', 'forecast', 'weather_advice', 'knowledge', 'air_quality'}
SYSTEM_PROMPT = 'You are AtmosIQ supervisor.'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using computation device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Loading base model {BASE_MODEL_NAME}...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto' if device == 'cuda' else None,
    trust_remote_code=True
)

print(f'Loading LoRA V2 Adapter from {ADAPTER_DIR}...')
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print('Model and Adapter successfully loaded!')


In [ ]:
# Step 5: Route Extraction and Normalization
def extract_route(text):
    clean = text.strip().lower()
    clean = re.sub(r'['"{}:;,.]', '', clean).strip()
    if clean in VALID_ROUTES:
        return clean
    tokens = clean.split()
    if tokens and tokens[0] in VALID_ROUTES:
        return tokens[0]
    for r in sorted(VALID_ROUTES, key=len, reverse=True):
        if r in clean:
            return r
    return None


In [ ]:
# Step 6: Execute Genuine Inference on 25 Challenge Prompts
predictions = []
correct_count = 0
route_counts = Counter()
route_correct = Counter()
confusion = defaultdict(Counter)

print('=== Running LoRA V2 Adapter Inference on 25 Challenge Prompts ===\n')

for item in CHALLENGE_DATA:
    prompt = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item["text"]}<|im_end|>\n<|im_start|>assistant\n'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    input_len = inputs['input_ids'].shape[1]
    raw_output = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    predicted_route = extract_route(raw_output)
    expected_route = item['expected_route']
    
    is_correct = (predicted_route == expected_route)
    if is_correct:
        correct_count += 1
        route_correct[expected_route] += 1
        
    route_counts[expected_route] += 1
    confusion[expected_route][predicted_route or 'invalid'] += 1
    
    status = 'PASS' if is_correct else 'FAIL'
    print(f'[{status}] ID {item["id"]:02d}: Expected: {expected_route:<15} | Predicted: {str(predicted_route):<15} | Raw: "{raw_output}" | Text: "{item["text"]}"')
    
    predictions.append({
        'id': item['id'],
        'text': item['text'],
        'category': item['category'],
        'expected': expected_route,
        'predicted': predicted_route,
        'raw_output': raw_output,
        'correct': is_correct,
        'notes': item['notes']
    })

accuracy = correct_count / len(CHALLENGE_DATA)
invalid_route_count = sum(1 for p in predictions if p['predicted'] is None or p['predicted'] not in VALID_ROUTES)

# Compute per-class precision, recall, F1
per_class_metrics = {}
macro_p, macro_r, macro_f1 = 0.0, 0.0, 0.0
for route in sorted(VALID_ROUTES):
    tp = route_correct[route]
    fn = route_counts[route] - tp
    fp = sum(confusion[other][route] for other in VALID_ROUTES if other != route)
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0.0
    per_class_metrics[route] = {
        'precision': round(p, 4),
        'recall': round(r, 4),
        'f1-score': round(f1, 4),
        'support': route_counts[route]
    }
    macro_p += p
    macro_r += r
    macro_f1 += f1

macro_avg = {
    'precision': round(macro_p / len(VALID_ROUTES), 4),
    'recall': round(macro_r / len(VALID_ROUTES), 4),
    'f1-score': round(macro_f1 / len(VALID_ROUTES), 4),
    'support': len(CHALLENGE_DATA)
}

report = {
    'model_name': BASE_MODEL_NAME,
    'adapter_dir': ADAPTER_DIR,
    'total_prompts': len(CHALLENGE_DATA),
    'correct_predictions': correct_count,
    'invalid_route_count': invalid_route_count,
    'accuracy': accuracy,
    'per_class_metrics': per_class_metrics,
    'macro_avg': macro_avg,
    'confusion_matrix': {k: dict(v) for k, v in confusion.items()},
    'predictions': predictions
}

print('\n' + '=' * 55)
print(f'LoRA V2 Challenge Score: {correct_count}/{len(CHALLENGE_DATA)} ({accuracy:.2%})')
print(f'Invalid Route Count: {invalid_route_count}')
print(f'Macro F1: {macro_avg["f1-score"]:.4f}')
print('=' * 55)


In [ ]:
# Step 7: Export Report and Predictions JSON & Download in Colab
with open('atmosiq_lora_v2_challenge_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

with open('atmosiq_lora_v2_challenge_predictions.json', 'w', encoding='utf-8') as f:
    json.dump(predictions, f, indent=2)

print('Saved files:')
print('- atmosiq_lora_v2_challenge_report.json')
print('- atmosiq_lora_v2_challenge_predictions.json')

try:
    from google.colab import files
    files.download('atmosiq_lora_v2_challenge_report.json')
    files.download('atmosiq_lora_v2_challenge_predictions.json')
except Exception as e:
    print('Automatic download failed or running outside Colab. Download directly from the left file pane.')
